# Research · Dehazing — Ablation Study & Quantitative Evaluation (NB 2/2)

**Group 01 · Dept. of CSE, East West University**

The **research** notebook. One-factor-at-a-time ablations from a fixed baseline (patch 15 · ω 0.95 · t₀ 0.10 ·
A=dcp_top · guided refine), stratified by haze density, plus a **failure analysis** on bright/sky regions.
Produces research-quality **tables** (CSV + rendered) and **figures**. Attach RESIDE/O-HAZE for reportable
numbers; otherwise the synthetic fallback runs.

In [ ]:
# research-figure style + palette (colourblind-safe)
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "axes.titleweight": "bold", "figure.autolayout": True})
PAL = {"blue": "#2f6db5", "orange": "#e07b39", "green": "#3f9b6d", "red": "#c0392b",
       "grey": "#7f8c8d", "purple": "#7d5ba6"}

In [ ]:
# ===== Dark Channel Prior — classical dehazing (pure NumPy/OpenCV) =====
import numpy as np, cv2

def dark_channel(img, patch=15):
    mn = img.min(axis=2)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    return cv2.erode(mn, k)

def atmospheric_light(img, dark, method="dcp_top"):
    h, w = dark.shape; flat_img = img.reshape(-1, 3)
    if method == "brightest":
        return flat_img[img.reshape(-1, 3).sum(1).argmax()]
    if method == "quadtree":
        reg = img
        while reg.shape[0] * reg.shape[1] > 400:
            H, W = reg.shape[:2]; hh, ww = H // 2, W // 2
            quads = [reg[:hh, :ww], reg[:hh, ww:], reg[hh:, :ww], reg[hh:, ww:]]
            scores = [q.reshape(-1, 3).mean() - q.reshape(-1, 3).std() for q in quads]
            reg = quads[int(np.argmax(scores))]
            if min(reg.shape[:2]) < 2: break
        fr = reg.reshape(-1, 3); return fr[fr.sum(1).argmax()]
    n = max(1, int(0.001 * h * w))                       # dcp_top: top-0.1% dark-channel pixels
    idx = np.argpartition(dark.ravel(), -n)[-n:]
    cand = flat_img[idx]; return cand[cand.sum(1).argmax()]

def guided_filter(guide, src, r=60, eps=1e-3):
    r = max(1, int(r)); box = lambda x: cv2.boxFilter(x, cv2.CV_64F, (r, r))
    mI, mp = box(guide), box(src); mIp = box(guide * src)
    cov = mIp - mI * mp; var = box(guide * guide) - mI * mI
    a = cov / (var + eps); b = mp - a * mI
    return box(a) * guide + box(b)

def transmission(img, A, omega=0.95, patch=15):
    return 1.0 - omega * dark_channel(img / np.maximum(A, 1e-6), patch)

def recover(img, A, t, t0=0.1):
    tt = np.clip(t, t0, 1.0)[..., None]
    return np.clip((img - A) / tt + A, 0.0, 1.0)

def dehaze(img, patch=15, omega=0.95, t0=0.1, refine="guided", A_method="dcp_top", r=60, eps=1e-3):
    dark = dark_channel(img, patch)
    A = atmospheric_light(img, dark, A_method)
    t_raw = transmission(img, A, omega, patch)
    if refine == "guided":
        gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float64) / 255.0
        t = guided_filter(gray, t_raw, r, eps)
    else:
        t = t_raw
    J = recover(img, A, t, t0)
    return {"dehazed": J, "dark": dark, "A": A, "t_raw": t_raw, "t": t}

In [ ]:
# ===== paired data: RESIDE/O-HAZE if attached, else synthetic-haze fallback =====
from pathlib import Path
from PIL import Image
import numpy as np

def _load(p, size=256):
    im = Image.open(p).convert("RGB").resize((size, size))
    return np.asarray(im, np.float64) / 255.0

def find_reside(maxn=60):
    root = Path("/kaggle/input")
    hazy_dir = next((d for d in root.rglob("*") if d.is_dir() and d.name.lower() in ("hazy", "haze")
                     and (any(d.glob("*.png")) or any(d.glob("*.jpg")))), None)
    if hazy_dir is None: return None
    gt_dir = next((d for d in hazy_dir.parent.iterdir() if d.is_dir() and
                   d.name.lower() in ("gt", "clear", "clean")), None)
    if gt_dir is None: return None
    gtmap = {}
    for g in list(gt_dir.glob("*.png")) + list(gt_dir.glob("*.jpg")):
        gtmap.setdefault("".join(ch for ch in g.stem if ch.isdigit()), g)
    pairs = []
    for hz in sorted(list(hazy_dir.glob("*.png")) + list(hazy_dir.glob("*.jpg"))):
        key = "".join(ch for ch in hz.stem if ch.isdigit()).split("_")[0] if hz.stem else ""
        key = "".join(ch for ch in hz.stem if ch.isdigit())
        g = gtmap.get(key) or next((v for k, v in gtmap.items() if key.startswith(k)), None)
        if g is not None:
            pairs.append((_load(hz), _load(g), "reside"))
        if len(pairs) >= maxn: break
    return pairs or None

def synth_haze(clear, beta, A_val):
    h, w = clear.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w]
    depth = 0.2 + 0.8 * np.sqrt(((xx - w/2)/w)**2 + ((yy - h/2)/h)**2)   # radial depth proxy
    t = np.exp(-beta * depth)[..., None]
    A = np.array(A_val).reshape(1, 1, 3)
    hazy = clear * t + A * (1 - t)
    return np.clip(hazy, 0, 1)

def synthetic_bank():
    from skimage import data as skd
    clears = []
    for name in ["astronaut", "coffee", "chelsea", "rocket", "logo"]:
        try:
            im = getattr(skd, name)()
            if im.ndim == 3 and im.shape[2] >= 3:
                clears.append(_load_arr(im))
        except Exception:
            pass
    pairs, DENS = [], {0.8: "light", 1.6: "medium", 2.6: "dense"}
    for c in clears:
        for beta in (0.8, 1.6, 2.6):
            for A in ([0.85, 0.88, 0.9], [0.75, 0.78, 0.82]):
                pairs.append((synth_haze(c, beta, A), c, DENS[beta]))
    return pairs

def _load_arr(im, size=256):
    from PIL import Image as I
    return np.asarray(I.fromarray(im[..., :3]).convert("RGB").resize((size, size)), np.float64) / 255.0

pairs = find_reside()
SOURCE = "RESIDE/O-HAZE (attached)" if pairs else "synthetic-haze fallback"
if not pairs:
    pairs = synthetic_bank()
print(f"eval source: {SOURCE} | {len(pairs)} paired (hazy, clear) images")

In [ ]:
# ===== full-reference metrics =====
from skimage.metrics import peak_signal_noise_ratio as _psnr, structural_similarity as _ssim
def scores(dehazed, gt):
    return (_psnr(gt, dehazed, data_range=1.0),
            _ssim(gt, dehazed, data_range=1.0, channel_axis=2))

In [ ]:
# ===== helpers: evaluate a config over the whole eval set + render a table figure =====
import numpy as np, pandas as pd
BASE = dict(patch=15, omega=0.95, t0=0.1, refine="guided", A_method="dcp_top")

def eval_config(**kw):
    cfg = {**BASE, **kw}; ps, ss = [], []
    for hz, gt, _ in pairs:
        o = dehaze(hz, **cfg); p, s = scores(o["dehazed"], gt); ps.append(p); ss.append(s)
    return np.mean(ps), np.std(ps), np.mean(ss), np.std(ss)

def table_figure(df, title, fname, highlight_cols=("PSNR", "SSIM")):
    fig, ax = plt.subplots(figsize=(min(12, 1.6*len(df.columns)), 0.5 + 0.4*len(df)))
    ax.axis("off"); tbl = ax.table(cellText=np.round(df.values, 4), colLabels=df.columns,
                                   rowLabels=df.index, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.4)
    for col in highlight_cols:
        if col in df.columns:
            j = list(df.columns).index(col); best = df[col].values.argmax()
            tbl[(best+1, j)].set_facecolor("#d6ecd6")
    for j in range(len(df.columns)): tbl[(0, j)].set_facecolor("#2f6db5"); tbl[(0, j)].set_text_props(color="w", weight="bold")
    ax.set_title(title, fontweight="bold", pad=12)
    plt.savefig(fname, bbox_inches="tight"); plt.show()
print("baseline config:", BASE, "| eval N =", len(pairs))

## 1. Ablation sweeps (one factor at a time) — research table

In [ ]:
# sweep each factor; collect into one tidy results table
sweeps = {
    "patch_size":   [("patch", v) for v in (3, 7, 15, 31)],
    "A_method":     [("A_method", v) for v in ("dcp_top", "brightest", "quadtree")],
    "omega":        [("omega", v) for v in (0.80, 0.90, 0.95, 0.99)],
    "t0":           [("t0", v) for v in (0.05, 0.10, 0.20)],
    "refinement":   [("refine", v) for v in ("none", "guided")],
}
records = []
for factor, levels in sweeps.items():
    for key, val in levels:
        mp, sp, ms, ss = eval_config(**{key: val})
        records.append({"factor": factor, "level": str(val), "PSNR": mp, "PSNR_sd": sp, "SSIM": ms, "SSIM_sd": ss})
abl = pd.DataFrame(records)
abl.to_csv("dehaze_ablation_results.csv", index=False)
print(abl.round(3).to_string(index=False))

# rendered table figure per factor (best PSNR/SSIM highlighted)
for factor in sweeps:
    sub = abl[abl.factor == factor].set_index("level")[["PSNR", "PSNR_sd", "SSIM", "SSIM_sd"]]
    table_figure(sub, f"Ablation — {factor}", f"table_ablation_{factor}.png")

## 2. Ablation curves (the headline: which stage moves the metric)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, factor, col in [(ax[0], "patch_size", PAL["blue"]), (ax[1], "omega", PAL["orange"]), (ax[2], "t0", PAL["green"])]:
    sub = abl[abl.factor == factor]
    xs = [float(x) for x in sub.level]
    a.errorbar(xs, sub.PSNR, yerr=sub.PSNR_sd, marker="o", color=col, capsize=3)
    a.set_xlabel(factor); a.set_ylabel("PSNR (dB)"); a.set_title(f"PSNR vs {factor}")
plt.suptitle("DCP parameter sensitivity (mean ± sd over eval set)", y=1.02)
plt.savefig("fig_ablation_curves.png", bbox_inches="tight"); plt.show()

# refinement + A-method as grouped bars (categorical factors)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for a, factor in [(ax[0], "refinement"), (ax[1], "A_method")]:
    sub = abl[abl.factor == factor]
    a.bar(sub.level, sub.PSNR, yerr=sub.PSNR_sd, color=PAL["purple"], capsize=3, alpha=.85)
    a.set_ylabel("PSNR (dB)"); a.set_title(factor); a.set_ylim(sub.PSNR.min()-1, sub.PSNR.max()+1)
plt.suptitle("Categorical-stage effect on PSNR", y=1.02)
plt.savefig("fig_ablation_categorical.png", bbox_inches="tight"); plt.show()

## 3. Haze-density stratification (H3: does DCP's gain shrink with density?)

In [ ]:
# best config (max PSNR level per factor) then stratify PSNR by density
best = {}
for factor, levels in sweeps.items():
    sub = abl[abl.factor == factor]; blevel = sub.loc[sub.PSNR.idxmax(), "level"]
    key = levels[0][0]
    best[key] = (int(blevel) if factor == "patch_size" else float(blevel) if factor in ("omega", "t0") else blevel)
print("best-per-factor config:", best)

by_dens = {}
for hz, gt, dens in pairs:
    o = dehaze(hz, **best); p, _ = scores(o["dehazed"], gt); by_dens.setdefault(dens, []).append(p)
order = [d for d in ["light", "medium", "dense", "reside"] if d in by_dens]
plt.figure(figsize=(7, 4.2))
plt.boxplot([by_dens[d] for d in order], labels=order, showmeans=True)
plt.ylabel("PSNR (dB)"); plt.title("PSNR by haze density (best DCP config)")
plt.savefig("fig_density_boxplot.png", bbox_inches="tight"); plt.show()
dens_tbl = pd.DataFrame({d: [np.mean(by_dens[d]), np.std(by_dens[d]), len(by_dens[d])] for d in order},
                        index=["mean_PSNR", "sd_PSNR", "n"]).T.round(3)
print(dens_tbl.to_string())

## 4. Failure analysis — bright/sky regions (H2: A-estimation drives failure)

In [ ]:
# error inside bright ("sky-like": high value, low saturation) regions vs elsewhere
import cv2
def bright_mask(hz):
    hsv = cv2.cvtColor((hz*255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float64)
    return (hsv[..., 2]/255.0 > 0.75) & (hsv[..., 1]/255.0 < 0.25)
rows = []
for hz, gt, dens in pairs:
    o = dehaze(hz, **best); err = np.abs(o["dehazed"] - gt).mean(2)
    m = bright_mask(hz)
    if m.sum() > 20:
        rows.append({"in_bright": err[m].mean(), "elsewhere": err[~m].mean(), "density": dens})
fail = pd.DataFrame(rows)
if len(fail):
    summ = fail[["in_bright", "elsewhere"]].mean()
    print(f"mean abs error  in bright/sky regions: {summ['in_bright']:.4f}"
          f"  vs elsewhere: {summ['elsewhere']:.4f}"
          f"  (ratio {summ['in_bright']/max(summ['elsewhere'],1e-6):.2f}x)")
    plt.figure(figsize=(5.5, 4))
    plt.bar(["bright / sky", "elsewhere"], [summ["in_bright"], summ["elsewhere"]],
            color=[PAL["red"], PAL["grey"]], alpha=.85)
    plt.ylabel("mean abs error"); plt.title("DCP failure concentrates in bright regions")
    plt.savefig("fig_failure_bright.png", bbox_inches="tight"); plt.show()
    fail.to_csv("dehaze_failure_analysis.csv", index=False)
else:
    print("no sufficiently bright regions found in this eval set.")

## 5. Best / worst cases + save summary

In [ ]:
# rank by PSNR under best config; show best 2 and worst 2
scored = []
for hz, gt, dens in pairs:
    o = dehaze(hz, **best); p, s = scores(o["dehazed"], gt); scored.append((p, s, hz, o["dehazed"], gt))
scored.sort(key=lambda z: z[0])
show = scored[:2] + scored[-2:]; tags = ["worst", "worst", "best", "best"]
fig, ax = plt.subplots(4, 3, figsize=(11, 13))
for r, (p, s, hz, jj, gt) in enumerate(show):
    for c, (im, t) in enumerate([(hz, f"{tags[r]} hazy"), (jj, f"DCP  PSNR {p:.1f}/SSIM {s:.2f}"), (gt, "GT")]):
        ax[r, c].imshow(im, vmin=0, vmax=1); ax[r, c].set_title(t, fontsize=8); ax[r, c].axis("off")
plt.suptitle("Best vs worst DCP results (best config)", y=1.0)
plt.savefig("fig_best_worst.png", bbox_inches="tight"); plt.show()

import json
summary = {"source": SOURCE, "eval_N": len(pairs), "baseline_cfg": BASE, "best_cfg": best,
           "best_cfg_mean_PSNR": float(np.mean([z[0] for z in scored])),
           "best_cfg_mean_SSIM": float(np.mean([z[1] for z in scored]))}
json.dump(summary, open("dehaze_results.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

## Summary
Stage-wise ablation attributes restoration quality to specific DCP operators (H1: refinement; H2: `A`-estimation
drives bright-region failure; H3: density interaction). All tables (`*.csv`, rendered `table_*.png`) and figures
(`fig_*.png`) are report-ready. Optional next notebook `dehaze-learned-reference` positions DCP against a deep
model on the identical metrics.